In [ ]:
import requests
from bs4 import BeautifulSoup, Tag
import json


def _extract_text(tag: Tag | None) -> str | None:
    """Return the text of tag. If it is None then return None."""
    return tag.get_text(" ", strip=True) if tag is not None else None

# url = 'https://ttb.utoronto.ca/'
url = 'https://artsci.calendar.utoronto.ca/search-courses'

headers = {
    "User-Agent": "Monalisa/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
r = requests.get(url, headers=headers)
print("HTTP Status Code:", r.status_code)

r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")
print("Title:", soup.title)
print("HTML has :", "ABP104Y1" in r.text)


last_page = soup.select_one('a[title="Go to last page"]')
href = last_page['href']
pages = int(''.join(c for c in href if c.isdigit()))


print(last_page)
print(pages)

all_course = []

for page in range(pages + 1):
    url = f'https://artsci.calendar.utoronto.ca/search-courses?page={page}'
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    courses = soup.select('.view-content > .views-row')
    for course in courses:

        title = _extract_text(
            course.select_one(
                '.js-views-accordion-group-header > div'
            )
        )

        assert title is not None and ' - ' in title, \
            f'Page: {page}\nInvalid title:{title}\nHTML: \n{course.prettify()}'

        code, name = title.split(' - ', 1)

        prereq = _extract_text(
            course.select_one(
                '.views-field-field-prerequisite .field-content'
            )
        )

        exclusion = _extract_text(
            course.select_one(
                '.views-field-field-exclusion .field-content'
            )
        )

        course_data = {
            'code': code,
            'name': name,
            'prereq': prereq,
            'exclusion': exclusion
        }
        
        all_course.append(course_data)

print(f'Total number of Art&Sci courses: {len(all_course)}')

with open('arisci_raw.json', 'w', encoding='utf-8') as f:
    json.dump(all_course, f, indent=4, ensure_ascii=False)




HTTP Status Code: 200
Title: <title>Course Search | Academic Calendar</title>
HTML has : True
<a href="?page=178" title="Go to last page">
<span class="visually-hidden">Last page</span>
<span aria-hidden="true">Last »</span>
</a>
178
Total number of Art&Sci courses: 5368
